In [ ]:
!pip install ultralytics split-folders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.8 MB/s eta 0:00:00


**MOUNT GOOGLE DRIVE**

In [ ]:
from google.colab import drive
import os
import splitfolders
from ultralytics import YOLO
import shutil

drive.mount('/content/drive')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive


**DEFINE PATHS**

In [ ]:
path_disease_original = '/content/drive/MyDrive/Colab Notebooks/Chilli Plant Dataset/Chili Leaf Disease Augmented Dataset/Chili Leaf Disease Augmented Dataset'
path_growth_original = '/content/drive/MyDrive/Colab Notebooks/Chilli Plant Dataset/Chili Growth Stage Augmented Dataset/Chili Growth Stage Augmented Dataset'

Temporary local paths for the split and formatted data

In [ ]:
disease_dataset = "/content/disease_dataset"
growth_dataset = "/content/growth_dataset"
save_path = '/content/drive/MyDrive/Colab Notebooks/Chilli Plant Dataset/Saved_Models'

Verify paths exist before proceeding

In [ ]:
if not os.path.exists(path_disease_original):
    raise FileNotFoundError(f"Cannot find disease path: {path_disease_original}. Please check spelling/Google Drive.")
if not os.path.exists(path_growth_original):
    raise FileNotFoundError(f"Cannot find growth path: {path_growth_original}. Please check spelling/Google Drive.")

os.makedirs(save_path, exist_ok=True)

PREPARE DATASET (Auto-split into Train/Val folders for YOLO Classification)

In [ ]:
print("Splitting Disease Dataset...")
# Splits original data into 80% training, 20% validation
splitfolders.ratio(path_disease_original, output=disease_dataset, seed=42, ratio=(0.8, 0.2))

print("Splitting Growth Dataset...")
splitfolders.ratio(path_growth_original, output=growth_dataset, seed=42, ratio=(0.8, 0.2))

Splitting Disease Dataset...


Copying files: 10212 files [05:23, 31.56 files/s] 


Splitting Growth Dataset...


Copying files: 8025 files [04:39, 28.67 files/s] 


**LOAD YOLO CLASSIFICATION MODELS**

In [ ]:
# These models are specifically designed for image classification and train MUCH faster
model_disease = YOLO("yolov8n-cls.pt")
model_growth = YOLO("yolov8n-cls.pt")

**Train Model**

In [9]:
print("\n--- Training Disease Model ---")
model_disease.train(data=disease_dataset, epochs=50, imgsz=224, device='cpu', patience=10)

print("\n--- Training Growth Model ---")
model_growth.train(data=growth_dataset, epochs=50, imgsz=224, device='cpu', patience=10)


--- Training Disease Model ---
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/disease_dataset, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f9d010fc830>
curves: []
curves_results: []
fitness: 1.0
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 1.0, 'metrics/accuracy_top5': 1.0, 'fitness': 1.0}
save_dir: PosixPath('/content/runs/classify/train-2')
speed: {'preprocess': 0.0015444259057349175, 'inference': 8.936708937113487, 'loss': 6.0549190700744833e-05, 'postprocess': 0.00011574719773151983}
top1: 1.0
top5: 1.0

SAVE THE BEST MODELS TO GOOGLE DRIVE

In [12]:
shutil.copy("runs/classify/train/weights/best.pt", f"{save_path}/chili_disease_yolo_cls.pt")
print("Disease YOLO Classification model saved")

shutil.copy("runs/classify/train-2/weights/best.pt", f"{save_path}/chili_growth_yolo_cls.pt")
print("Growth YOLO Classification model saved")

Disease YOLO Classification model saved
Growth YOLO Classification model saved


PREDICTION FUNCTION

In [15]:
def final_predict(img_path):
    print("-" * 40)
    print(f"FILE: {os.path.basename(img_path)}")

    # Load our custom trained models
    disease_predictor = YOLO(f"{save_path}/chili_disease_yolo_cls.pt")
    growth_predictor = YOLO(f"{save_path}/chili_growth_yolo_cls.pt")

    d_results = disease_predictor(img_path)
    g_results = growth_predictor(img_path)

    # Extract classification top1 results
    d_prob = d_results[0].probs
    d_top1_index = d_prob.top1
    d_top1_conf = d_prob.top1conf.item()
    d_class_name = d_results[0].names[d_top1_index]

    g_prob = g_results[0].probs
    g_top1_index = g_prob.top1
    g_top1_conf = g_prob.top1conf.item()
    g_class_name = g_results[0].names[g_top1_index]

    print(f"DISEASE PREDICTION: {d_class_name} ({d_top1_conf * 100:.2f}%)")
    print(f"GROWTH PREDICTION: {g_class_name} ({g_top1_conf * 100:.2f}%)")
    print("-" * 40)

In [27]:
from google.colab import files
uploaded = files.upload()

for filename in uploaded.keys():
    test_image_path = "/content/" + filename
    final_predict(test_image_path)

Saving Bacterial Spot00001_sheared_5.jpg to Bacterial Spot00001_sheared_5 (1).jpg
----------------------------------------
FILE: Bacterial Spot00001_sheared_5 (1).jpg

image 1/1 /content/Bacterial Spot00001_sheared_5 (1).jpg: 224x224 Bacterial Spot 1.00, Cercospora Leaf Spot 0.00, Curl Virus 0.00, Nutrition Deficiency 0.00, White spot 0.00, 16.3ms
Speed: 10.4ms preprocess, 16.3ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)

image 1/1 /content/Bacterial Spot00001_sheared_5 (1).jpg: 224x224 Flower 0.99, Rotten Chili 0.01, Red Chili 0.00, Green Chili 0.00, Dry chili 0.00, 18.5ms
Speed: 7.5ms preprocess, 18.5ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
DISEASE PREDICTION: Bacterial Spot (100.00%)
GROWTH PREDICTION: Flower (99.48%)
----------------------------------------
